In [ ]:
from google.colab import files
files.upload()


In [ ]:
from google.colab import files
files.upload()


In [ ]:
from google.colab import files
files.upload()


In [ ]:
from google.colab import files
files.upload()


In [ ]:
# ============================================
# Fitbit Health RAG Chatbot
# Author: Aya Rizk
# Description: RAG chatbot answering questions
# about Fitbit health data using LangChain,
# ChromaDB, and Llama 3 via Groq
# ============================================

# Install packages
!pip install -q langchain langchain-core langchain-groq langchain-chroma langchain-community langchain-text-splitters chromadb sentence-transformers

# Import libraries
import os
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Set API key
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY_HERE"

# Load data
df_activity = pd.read_csv('dailyActivity_merged.csv')
df_sleep = pd.read_csv('minuteSleep_merged.csv')
df_weight = pd.read_csv('weightLogInfo_merged.csv')
print("✓ Data loaded!")

# Prepare documents
docs = [
    Document(page_content=df_activity.head(10).to_string(), metadata={"source": "activity"}),
    Document(page_content=df_sleep.head(10).to_string(), metadata={"source": "sleep"}),
    Document(page_content=df_weight.head(10).to_string(), metadata={"source": "weight"}),
]
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=30)
split_docs = splitter.split_documents(docs)
print("✓ Documents ready:", len(split_docs), "chunks")

# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
print("✓ Vector store created!")

# Build chatbot
prompt = ChatPromptTemplate.from_template("""
Answer the question based on the following Fitbit health data:
{context}

Question: {question}

Provide a clear and helpful answer based on the data above.
""")

llm = ChatGroq(model="llama-3.1-8b-instant")
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)
print("✓ Fitbit Health Chatbot is ready!")

# Test with questions
questions = [
    "What was the average steps per day?",
    "How many calories did users burn on average?",
    "What is the relationship between steps and calories burned?"
]

for q in questions:
    response = rag_chain.invoke(q)
    print(f"\nQ: {q}")
    print(f"A: {response.content}")
